In [3]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import folium
import warnings
warnings.filterwarnings('ignore')

# Target Area Settings
# Start with a Canton of Geneva
PLACE_NAME = "Canton of Geneva, Switzerland"

# Fetch Migros and Competitor Store Locations (Fixing Missing Data)
print("Fetching all supermarket and convenience store coordinates...")

# Fetch both large supermarkets and small convenience stores
tags = {'shop': ['supermarket', 'convenience']}
supermarkets = ox.features_from_place(PLACE_NAME, tags=tags)

# Transform building shapes (Polygon) into points (Point) for easier plotting and calculation
supermarkets['geometry'] = supermarkets['geometry'].centroid
supermarkets.head()

Fetching all supermarket and convenience store coordinates...


geometry         shop        addr:city  \
element id                                                                  
node    322007552  POINT (6.12107 46.16901)  supermarket  Plan-les-Ouates   
        322007712  POINT (6.16414 46.19206)  supermarket           Genève   
        322007998  POINT (6.16128 46.19331)  supermarket           Genève   
        322008034  POINT (6.15248 46.19086)  supermarket              NaN   
        322008463  POINT (6.18575 46.16695)  convenience              NaN   

                  addr:housenumber addr:postcode          addr:street   brand  \
element id                                                                      
node    322007552                2          1228        Chem. de Vers    Coop   
        322007712               86           NaN  Route de Florissant  Migros   
        322007998               56          1206  Route de Florissant    Coop   
        322008034              NaN          1206                  NaN  Migros   
        322008463              NaN           NaN                  NaN     NaN   

                  brand:wikidata    brand:wikipedia contact:phone  ...  \
element id                                                         ...   
node    322007552        Q432564  de:Coop (Schweiz)           NaN  ...   
        322007712        Q680727          en:Migros           NaN  ...   
        322007998        Q432564                NaN           NaN  ...   
        322008034        Q680727          en:Migros           NaN  ...   
        322008463            NaN                NaN           NaN  ...   

                  building:levels fuel:adblue fuel:diesel fuel:octane_95  \
element id                                                                 
node    322007552             NaN         NaN         NaN            NaN   
        322007712             NaN         NaN         NaN            NaN   
        322007998             NaN         NaN         NaN            NaN   
        322008034             NaN         NaN         NaN            NaN   
        322008463             NaN         NaN         NaN            NaN   

                  fuel:octane_98 self_service payment:visa_debit  \
element id                                                         
node    322007552            NaN          NaN                NaN   
        322007712            NaN          NaN                NaN   
        322007998            NaN          NaN                NaN   
        322008034            NaN          NaN                NaN   
        322008463            NaN          NaN                NaN   

                  payment:visa_electron room type  
element id                                         
node    322007552                   NaN  NaN  NaN  
        322007712                   NaN  NaN  NaN  
        322007998                   NaN  NaN  NaN  
        322008034                   NaN  NaN  NaN  
        322008463                   NaN  NaN  NaN  

[5 rows x 125 columns]

In [7]:
print(supermarkets.columns.tolist())

['geometry', 'shop', 'addr:city', 'addr:housenumber', 'addr:postcode', 'addr:street', 'brand', 'brand:wikidata', 'brand:wikipedia', 'contact:phone', 'contact:website', 'name', 'opening_hours', 'operator', 'operator:wikidata', 'payment:coins', 'payment:mastercard', 'payment:notes', 'payment:visa', 'wheelchair', 'stroller', 'phone', 'website', 'payment:cards', 'payment:cash', 'payment:contactless', 'payment:credit_cards', 'payment:debit_cards', 'cash_withdrawal', 'cash_withdrawal:fee', 'cash_withdrawal:purchase_required', 'cash_withdrawal:type', 'fixme', 'addr:city:en', 'addr:country', 'brand:wikipedia:de', 'level', 'opening_hours:url', 'operator:ref:CH:UID', 'ref:CH-GE:REG', 'check_date', 'branch', 'email', 'description', 'internet_access', 'name:de', 'name:en', 'name:fr', 'wheelchair:description', 'check_date:opening_hours', 'post_office', 'post_office:brand', 'post_office:website', 'payment:maestro', 'not:brand:wikidata', 'name:pt', 'name:ru', 'tobacco', 'addr:floor', 'bulk_purchase',

In [4]:
# Function to categorize supermarket brands
def categorize_brand(name):
    if pd.isna(name): 
        return 'Other'
    name_lower = str(name).lower()
    
    # Comprehensive check: e.g., 'migrolino' or 'voi' will also be grouped under 'Migros'
    if 'migros' in name_lower or 'migrolino' in name_lower or 'voi' in name_lower: 
        return 'Migros'
    elif 'coop' in name_lower: 
        return 'Coop'
    elif 'denner' in name_lower: 
        return 'Denner'
    elif 'aldi' in name_lower: 
        return 'Aldi'
    elif 'lidl' in name_lower: 
        return 'Lidl'
    else: 
        return 'Other'

# Prevent Errors in case a specific area doesn't have these tag columns at all
if 'brand' not in supermarkets.columns: 
    supermarkets['brand'] = None
if 'name' not in supermarkets.columns: 
    supermarkets['name'] = None
if 'operator' not in supermarkets.columns: 
    supermarkets['operator'] = None

# Look for brand names across 3 fallback columns (Brand -> Store Name -> Operator)
# If the first one is missing, it will fetch the next available option
supermarkets['search_name'] = supermarkets['brand'].fillna(supermarkets['name']).fillna(supermarkets['operator'])

# Apply the categorization function to the consolidated name column
supermarkets['brand_category'] = supermarkets['search_name'].apply(categorize_brand)

# Filter for the top 5 competing giant brands
target_brands = ['Migros', 'Coop', 'Denner', 'Aldi', 'Lidl']
main_stores = supermarkets[supermarkets['brand_category'].isin(target_brands)]

# Print the total number of branches found after code improvements
print("Total branches retrieved after code optimization:")
print(main_stores['brand_category'].value_counts())

Total branches retrieved after code optimization:
brand_category
Coop      52
Migros    41
Denner    28
Aldi       7
Lidl       5
Name: count, dtype: int64


In [5]:
main_stores.head()

geometry         shop        addr:city  \
element id                                                                  
node    322007552  POINT (6.12107 46.16901)  supermarket  Plan-les-Ouates   
        322007712  POINT (6.16414 46.19206)  supermarket           Genève   
        322007998  POINT (6.16128 46.19331)  supermarket           Genève   
        322008034  POINT (6.15248 46.19086)  supermarket              NaN   
        447808467  POINT (6.11253 46.23277)  supermarket              NaN   

                  addr:housenumber addr:postcode          addr:street   brand  \
element id                                                                      
node    322007552                2          1228        Chem. de Vers    Coop   
        322007712               86           NaN  Route de Florissant  Migros   
        322007998               56          1206  Route de Florissant    Coop   
        322008034              NaN          1206                  NaN  Migros   
        447808467              NaN           NaN                  NaN  Migros   

                  brand:wikidata    brand:wikipedia contact:phone  ...  \
element id                                                         ...   
node    322007552        Q432564  de:Coop (Schweiz)           NaN  ...   
        322007712        Q680727          en:Migros           NaN  ...   
        322007998        Q432564                NaN           NaN  ...   
        322008034        Q680727          en:Migros           NaN  ...   
        447808467        Q680727          fr:Migros           NaN  ...   

                  fuel:diesel fuel:octane_95 fuel:octane_98 self_service  \
element id                                                                 
node    322007552         NaN            NaN            NaN          NaN   
        322007712         NaN            NaN            NaN          NaN   
        322007998         NaN            NaN            NaN          NaN   
        322008034         NaN            NaN            NaN          NaN   
        447808467         NaN            NaN            NaN          NaN   

                  payment:visa_debit payment:visa_electron room type  \
element id                                                             
node    322007552                NaN                   NaN  NaN  NaN   
        322007712                NaN                   NaN  NaN  NaN   
        322007998                NaN                   NaN  NaN  NaN   
        322008034                NaN                   NaN  NaN  NaN   
        447808467                NaN                   NaN  NaN  NaN   

                  search_name brand_category  
element id                                    
node    322007552        Coop           Coop  
        322007712      Migros         Migros  
        322007998        Coop           Coop  
        322008034      Migros         Migros  
        447808467      Migros         Migros  

[5 rows x 127 columns]

In [6]:
# Export to CSV Data with Address

# Reset Index
export_df = main_stores.reset_index()

# Extract Latitude and Longitude coordinates
export_df['latitude'] = export_df.geometry.y
export_df['longitude'] = export_df.geometry.x

# Define the list of address and details columns to include
desired_columns = [
    'osmid', 'name', 'brand_category', 'operator', 'shop', 
    'latitude', 'longitude', 'addr:housenumber', 'addr:street', 
    'addr:postcode', 'addr:city'
]

# Check and filter only for columns that actually exist (prevents errors if some areas lack address tags)
final_columns = [col for col in desired_columns if col in export_df.columns]
clean_df = export_df[final_columns]

# Save to CSV file
file_name = 'geneva_supermarkets_data_with_address.csv'
clean_df.to_csv(file_name, index=False, encoding='utf-8-sig')

print(f"File successfully exported with address details!")
display(clean_df.head())

File successfully exported with address details!


,name,brand_category,operator,shop,latitude,longitude,addr:housenumber,addr:street,addr:postcode,addr:city
0,Coop,Coop,NaN,supermarket,46.169006,6.121073,2,Chem. de Vers,1228,Plan-les-Ouates
1,Migros,Migros,NaN,supermarket,46.192060,6.164144,86,Route de Florissant,NaN,Genève
2,COOP,Coop,NaN,supermarket,46.193306,6.161276,56,Route de Florissant,1206,Genève
3,Migros,Migros,NaN,supermarket,46.190858,6.152480,NaN,NaN,1206,NaN
4,Migros,Migros,Société Coopérative Migros Genève,supermarket,46.232767,6.112525,NaN,NaN,NaN,NaN


In [11]:
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import FeatureGroupSubGroup
from IPython.display import display

print("Loading Data...")
# Load Switzerland GeoJSON boundary file
swiss_boundaries = gpd.read_file('switzerland.geojson')

# Load the exported supermarket CSV file
stores_df = pd.read_csv('geneva_supermarkets_data_with_address.csv')

print("Preparing Geospatial Data...")
# Convert DataFrame to GeoDataFrame and set Coordinate Reference System (CRS) to WGS84
stores_gdf = gpd.GeoDataFrame(
    stores_df, 
    geometry=gpd.points_from_xy(stores_df.longitude, stores_df.latitude),
    crs="EPSG:4326"
)

# Verify and match the CRS of the boundary map to WGS84
if swiss_boundaries.crs is None or swiss_boundaries.crs.to_epsg() != 4326:
    swiss_boundaries = swiss_boundaries.to_crs(epsg=4326)

print("Performing Spatial Join (Matching stores into municipal boundaries)...")
joined_data = gpd.sjoin(stores_gdf, swiss_boundaries, how="inner", predicate="intersects")

# Display the first 5 rows in Jupyter
display(joined_data.head())

print("Creating and Generating Map with Hover & Legend...")
# Calculate the map center coordinates
center_lat = stores_df['latitude'].mean()
center_lon = stores_df['longitude'].mean()

# Create the base map
m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="cartodbpositron")

# Define brand color mapping
brand_colors = {
    'Migros': 'orange',
    'Coop': 'red',
    'Denner': 'darkred',
    'Aldi': 'blue',
    'Lidl': 'purple'  # Changed to purple for better visual contrast on a light map
}

# Create a master group for stores and sub-groups for each brand to serve as an interactive Legend
main_group = folium.FeatureGroup(name="Supermarkets")
m.add_child(main_group)

# Create subgroups dynamically based on defined brands
subgroups = {}
for brand, color in brand_colors.items():
    subgroups[brand] = FeatureGroupSubGroup(main_group, f"{brand} ({color.capitalize()})")
    m.add_child(subgroups[brand])

# Loop through each store to plot on the map
for idx, row in joined_data.iterrows():
    brand = row['brand_category']
    color = brand_colors.get(brand, 'gray')
    
    # Pop-up text (Appears when clicked)
    popup_text = f"<b>Brand:</b> {brand}<br><b>Name:</b> {row.get('name', 'N/A')}<br><b>Street:</b> {row.get('addr:street', 'N/A')}"
    
    # Hover text / Tooltip (Appears instantly when mouse hovers over the marker)
    hover_text = f"{brand}: {row.get('name', 'Unnamed Store')}"
    
    # Create the CircleMarker
    marker = folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        popup=folium.Popup(popup_text, max_width=300),
        tooltip=hover_text,  # Enables the hover detail feature
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8
    )
    
    # Add to its corresponding brand subgroup, or directly to map if 'Other'
    if brand in subgroups:
        marker.add_to(subgroups[brand])
    else:
        marker.add_to(m)

# Add Layer Control to act as the interactive Map Legend panel (Top-Right corner)
folium.LayerControl(collapsed=False).add_to(m)

# Display the map
m

Loading Data...
Preparing Geospatial Data...
Performing Spatial Join (Matching stores into municipal boundaries)...


,name_left,brand_category,operator,shop,latitude,longitude,addr:housenumber,addr:street,addr:postcode,addr:city,geometry,index_right,name_right,cartodb_id,created_at,updated_at
0,Coop,Coop,NaN,supermarket,46.169006,6.121073,2,Chem. de Vers,1228.0,Plan-les-Ouates,POINT (6.12107 46.16901),24,Genève,25,2013-12-01 19:38:46+01:00,2013-12-01 19:38:47+01:00
1,Migros,Migros,NaN,supermarket,46.192060,6.164144,86,Route de Florissant,NaN,Genève,POINT (6.16414 46.19206),24,Genève,25,2013-12-01 19:38:46+01:00,2013-12-01 19:38:47+01:00
2,COOP,Coop,NaN,supermarket,46.193306,6.161276,56,Route de Florissant,1206.0,Genève,POINT (6.16128 46.19331),24,Genève,25,2013-12-01 19:38:46+01:00,2013-12-01 19:38:47+01:00
3,Migros,Migros,NaN,supermarket,46.190858,6.152480,NaN,NaN,1206.0,NaN,POINT (6.15248 46.19086),24,Genève,25,2013-12-01 19:38:46+01:00,2013-12-01 19:38:47+01:00
4,Migros,Migros,Société Coopérative Migros Genève,supermarket,46.232767,6.112525,NaN,NaN,NaN,NaN,POINT (6.11253 46.23277),24,Genève,25,2013-12-01 19:38:46+01:00,2013-12-01 19:38:47+01:00


Creating and Generating Map with Hover & Legend...
